In [1]:
# /// script
# requires-python = ">=3.11"
# dependencies = [
#     "neurocnl",
#     "snntorch==0.9.4",
#     "torch==2.13.0",
# ]
# ///

# NeuroMorphic Pipeline — Snntorch Sim

Generated 2026-07-19 06:38 UTC.

**Architecture:** defined in the Architecture tab (CNL spec below).
**Pipeline config:** edit `config` in the next cell to change training parameters.

In [2]:
# ── Pipeline configuration ───────────────────────────────────────────
# Workspace settings used when this notebook was generated.

config = {
    "dataset":   "dataLoader_1784115155942_ds_test.pt",
    "framework": "snntorch_sim",
}

print('Config loaded:', config)

Config loaded: {'dataset': 'dataLoader_1784115155942_ds_test.pt', 'framework': 'snntorch_sim'}


In [3]:
import json as _json


def _nmtk_emit(
    epoch: int, total: int, loss: float, accuracy: float, layer_rates: dict
) -> None:
    print(
        _json.dumps(
            {
                "__nmtk_progress__": True,
                "epoch": epoch,
                "total_epochs": total,
                "loss": loss,
                "accuracy": accuracy,
                "layer_spike_rates": layer_rates,
            }
        ),
        flush=True,
    )


In [4]:
import torch
from torch.serialization import safe_globals
from torch.utils.data import DataLoader, TensorDataset

try:
    with safe_globals([TensorDataset]):
        _raw = torch.load('dataLoader_1784115155942_ds_test.pt', weights_only=True)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Dataset file not found: ' + 'dataLoader_1784115155942_ds_test.pt' + '. '
        'This file does not exist yet — if it is a hand-built stimulus/dataset, generate it first (see the notebook guide\'s data-preparation step), then retry.'
    ) from exc
except Exception as exc:
    raise RuntimeError(
        'Could not safely load trusted .pt dataset ' + 'dataLoader_1784115155942_ds_test.pt' + '. '
        'Expected a TensorDataset, tuple/list of tensors, a bare tensor, or a dict with data/labels. '
        'Regenerate the dataset in one of those formats, then retry.'
    ) from exc
# Handle TensorDataset, dict, raw tuple, or a bare tensor (e.g. an
# unlabeled stimulus/spike-train tensor with no dataset wrapper)
_ds = _raw if hasattr(_raw, 'tensors') else TensorDataset(*_raw) if isinstance(_raw, (tuple, list)) else TensorDataset(_raw) if torch.is_tensor(_raw) else TensorDataset(_raw['data'], _raw['labels'])

train_loader = DataLoader(_ds, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(_ds, batch_size=32, shuffle=False,          num_workers=0)
print(f'.pt dataset: {len(_ds)} samples, batch_size=32')

.pt dataset: 140 samples, batch_size=32


## Architecture

Network compiled from CNL spec via NIR.

In [5]:
# CNL spec — auto-generated from the Architecture canvas tab.
# To change the network, edit the Architecture tab and regenerate.
cnl_spec = '''
Define a network named graph.
# Network with 1 input, 4 hidden nodes, 1 output.
# flow: nir.Input_1784102108810 → nir.Linear_1784102121480 → cnl.RSynaptic_1784123413089 → nir.Linear_1784123425347 → cnl.Synaptic_1784123439941 → nir.Output_1784123447064

# Layers:
Define an input port named nir.Input_1784102108810 with shape (12,).
Define a linear transformation named nir.Linear_1784102121480 with weight matrix shape (40, 12).
Define a RSynaptic neuron named cnl.RSynaptic_1784123413089 with neuron count 40, synaptic decay 0.75, membrane decay 0.85, and firing threshold 1.0.
Define a linear transformation named nir.Linear_1784123425347 with weight matrix shape (7, 40).
Define a Synaptic neuron named cnl.Synaptic_1784123439941 with neuron count 7, synaptic decay 0.45, membrane decay 0.7, and firing threshold 1.0.
Define an output port named nir.Output_1784123447064 with shape (7,).

# Connections:
nir.Input_1784102108810 connects to nir.Linear_1784102121480.
nir.Linear_1784102121480 connects to cnl.RSynaptic_1784123413089.
cnl.RSynaptic_1784123413089 connects to nir.Linear_1784123425347.
nir.Linear_1784123425347 connects to cnl.Synaptic_1784123439941.
cnl.Synaptic_1784123439941 connects to nir.Output_1784123447064.

'''

from neurocnl.compile import compile_to_nir

graph = compile_to_nir(cnl_spec)
print(f'Network: {len(graph.nodes)} nodes, {len(graph.edges)} edges')

Network: 6 nodes, 5 edges


In [6]:
import torch
import numpy as np
torch.manual_seed(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

"""snnTorch network — auto-generated from NIR graph."""

import torch
import torch.nn as nn
import snntorch as snn
import numpy as np

from snntorch import surrogate
spike_grad = surrogate.fast_sigmoid(slope=5.0)

_w = np.load('weights_snntorch_sim_d4e0f212.npz')  # weights file saved alongside this notebook

_expected_weight_keys = ['nir_linear_1784102121480_weight', 'nir_linear_1784123425347_weight']
_missing_weight_keys = [k for k in _expected_weight_keys if k not in _w.files]
if _missing_weight_keys:
    raise RuntimeError(
        f"'weights_snntorch_sim_d4e0f212.npz' is missing {_missing_weight_keys} — this weights "
        "file does not match the current network. Regenerate the notebook from "
        "the Architecture tab so its weights file matches this architecture."
    )

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Linear layer: 'nir.Linear_1784102121480'  shape (40, 12)
        self.nir_linear_1784102121480 = nn.Linear(12, 40)
        # weight is all-zeros in NIR graph; keeping PyTorch default init
        self.nir_linear_1784102121480.bias = None
        # RSynaptic population: 'cnl.RSynaptic_1784123413089'  (40 neurons, alpha=0.7500, beta=0.8500)
        self.cnl_rsynaptic_1784123413089 = snn.RSynaptic(alpha=0.750000, beta=0.850000, linear_features=40, threshold=1.0000, reset_mechanism='subtract', reset_delay=False, spike_grad=spike_grad)
        self.cnl_rsynaptic_1784123413089.recurrent.bias = None
        # Linear layer: 'nir.Linear_1784123425347'  shape (7, 40)
        self.nir_linear_1784123425347 = nn.Linear(40, 7)
        # weight is all-zeros in NIR graph; keeping PyTorch default init
        self.nir_linear_1784123425347.bias = None
        # Synaptic population: 'cnl.Synaptic_1784123439941'  (7 neurons, alpha=0.4500, beta=0.7000)
        self.cnl_synaptic_1784123439941 = snn.Synaptic(alpha=0.450000, beta=0.700000, threshold=1.0000, reset_mechanism='subtract', reset_delay=False, spike_grad=spike_grad)

    def forward(self, x):
        x = x.swapaxes(0, 1)  # (B,T,...) → (T,B,...)
        # initialise explicit hidden states
        spk_cnl_rsynaptic_1784123413089 = torch.zeros(x.shape[1], 40, dtype=x.dtype, device=x.device)
        syn_cnl_rsynaptic_1784123413089 = torch.zeros(x.shape[1], 40, dtype=x.dtype, device=x.device)
        mem_cnl_rsynaptic_1784123413089 = torch.zeros(x.shape[1], 40, dtype=x.dtype, device=x.device)
        syn_cnl_synaptic_1784123439941, mem_cnl_synaptic_1784123439941 = self.cnl_synaptic_1784123439941.init_synaptic()
        spk_rec, hid_rec = [], []
        for t in range(x.shape[0]):
            xt = x[t]
            xt = self.nir_linear_1784102121480(xt)
            spk_cnl_rsynaptic_1784123413089, syn_cnl_rsynaptic_1784123413089, mem_cnl_rsynaptic_1784123413089 = self.cnl_rsynaptic_1784123413089(xt, spk_cnl_rsynaptic_1784123413089, syn_cnl_rsynaptic_1784123413089, mem_cnl_rsynaptic_1784123413089)
            hid_rec.append(spk_cnl_rsynaptic_1784123413089)
            xt = spk_cnl_rsynaptic_1784123413089
            xt = self.nir_linear_1784123425347(xt)
            spk_cnl_synaptic_1784123439941, syn_cnl_synaptic_1784123439941, mem_cnl_synaptic_1784123439941 = self.cnl_synaptic_1784123439941(xt, syn_cnl_synaptic_1784123439941, mem_cnl_synaptic_1784123439941)
            spk_rec.append(spk_cnl_synaptic_1784123439941)
            xt = spk_cnl_synaptic_1784123439941
        _hid = torch.stack(hid_rec, dim=0) if hid_rec else torch.zeros_like(spk_rec[0]).unsqueeze(0)
        return torch.stack(spk_rec, dim=0), _hid  # (T,B,out), (T,B,hid)


net = Net().float()  # ponytail: npz weights load as float64; cast to match DataLoader float32 input
print(f'Net: {sum(p.numel() for p in net.parameters())} parameters')

Net: 2360 parameters


## Train

In [7]:
import torch
from torch.serialization import safe_globals
from torch.utils.data import DataLoader, TensorDataset

try:
    with safe_globals([TensorDataset]):
        _raw = torch.load('dataLoader_1784102160669_ds_train.pt', weights_only=True)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Dataset file not found: ' + 'dataLoader_1784102160669_ds_train.pt' + '. '
        'This file does not exist yet — if it is a hand-built stimulus/dataset, generate it first (see the notebook guide\'s data-preparation step), then retry.'
    ) from exc
except Exception as exc:
    raise RuntimeError(
        'Could not safely load trusted .pt dataset ' + 'dataLoader_1784102160669_ds_train.pt' + '. '
        'Expected a TensorDataset, tuple/list of tensors, a bare tensor, or a dict with data/labels. '
        'Regenerate the dataset in one of those formats, then retry.'
    ) from exc
# Handle TensorDataset, dict, raw tuple, or a bare tensor (e.g. an
# unlabeled stimulus/spike-train tensor with no dataset wrapper)
_ds = _raw if hasattr(_raw, 'tensors') else TensorDataset(*_raw) if isinstance(_raw, (tuple, list)) else TensorDataset(_raw) if torch.is_tensor(_raw) else TensorDataset(_raw['data'], _raw['labels'])

train_loader = DataLoader(_ds, batch_size=64, shuffle=True, num_workers=0)
test_loader  = DataLoader(_ds, batch_size=64, shuffle=False,          num_workers=0)
print(f'.pt dataset: {len(_ds)} samples, batch_size=64')
num_steps = 256
optimizer = torch.optim.AdamW(net.parameters(), lr=0.001, weight_decay=0.01)
_vl_every_n_epochs = 1
_vl_save_best_checkpoint = True
_vl_checkpoint_metric = 'val_accuracy'
_vl_checkpoint_mode = 'max'
_vl_best = float('-inf')
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=15, min_lr=0.0
)
import torch
from torch.serialization import safe_globals
from torch.utils.data import DataLoader, TensorDataset

try:
    with safe_globals([TensorDataset]):
        _val_raw = torch.load('dataLoader_1784105847372_ds_val.pt', weights_only=True)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Validation dataset file not found: ' + 'dataLoader_1784105847372_ds_val.pt' + '. '
        'Generate it first, then retry.'
    ) from exc
except Exception as exc:
    raise RuntimeError(
        'Could not safely load trusted .pt validation dataset ' + 'dataLoader_1784105847372_ds_val.pt' + '. '
        'Expected a TensorDataset, tuple/list of tensors, a bare tensor, or a dict with data/labels.'
    ) from exc
_val_ds = _val_raw if hasattr(_val_raw, 'tensors') else TensorDataset(*_val_raw) if isinstance(_val_raw, (tuple, list)) else TensorDataset(_val_raw) if torch.is_tensor(_val_raw) else TensorDataset(_val_raw['data'], _val_raw['labels'])

val_loader = DataLoader(_val_ds, batch_size=64, shuffle=False, num_workers=0)
print(f'.pt validation dataset: {len(_val_ds)} samples, batch_size=64')
net.train()
for epoch in range(700):
    _epoch_loss_sum = 0.0; _epoch_batches = 0
    for batch_idx, (data, targets) in enumerate(train_loader):
        optimizer.zero_grad()
        # Reset network membrane potentials
        for layer in net.modules():
            if hasattr(layer, 'reset_mem'):
                layer.reset_mem()
        spk_out, mem_out = net(data)
        import snntorch.functional as SF
        loss_fn = SF.ce_count_loss()
        loss_val = loss_fn(spk_out, targets)
        # ponytail: target_layer='cnl.RSynaptic' does not match a known variable (spk_out/mem_out); treating as hidden layer -> mem_out
        # L1 spike regularization: mean per-unit spike count over time and batch
        loss_val = loss_val + 0.001 * torch.mean(torch.sum(mem_out, dim=0))
        # ponytail: target_layer='cnl.RSynaptic' does not match a known variable (spk_out/mem_out); treating as hidden layer -> mem_out
        # L2 spike regularization: mean squared per-sample total spike count
        loss_val = loss_val + 1e-06 * torch.mean(torch.sum(torch.sum(mem_out, dim=0), dim=1) ** 2)
        loss_val.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        optimizer.step()
        _epoch_loss_sum += loss_val.item(); _epoch_batches += 1
    avg_loss = _epoch_loss_sum / _epoch_batches if _epoch_batches else 0.0
    scheduler.step(avg_loss)
    _nmtk_emit(
        epoch=epoch + 1,
        total=700,
        loss=avg_loss,
        accuracy=None,
        layer_rates={'output': float(spk_out.float().mean().item())},
    )
    if (epoch + 1) % _vl_every_n_epochs == 0:
        net.eval()
        _vl_correct = 0
        _vl_total = 0
        _vl_loss_sum = 0.0
        _vl_batches = 0
        with torch.no_grad():
            for _vl_data, _vl_targets in val_loader:
                _vl_spk_out, _vl_mem_out = net(_vl_data)
                _vl_loss_sum += loss_fn(_vl_spk_out, _vl_targets).item(); _vl_batches += 1
                _vl_correct += (_vl_spk_out.sum(0).argmax(1) == _vl_targets).sum().item()
                _vl_total += _vl_targets.size(0)
        val_loss = _vl_loss_sum / _vl_batches if _vl_batches else 0.0
        val_accuracy = _vl_correct / _vl_total if _vl_total else 0.0
        _vl_metric_value = (
            val_accuracy if _vl_checkpoint_metric == 'val_accuracy'
            else (val_loss if val_loss is not None else val_accuracy)
        )
        _vl_improved = (
            _vl_metric_value >= _vl_best if _vl_checkpoint_mode == 'max'
            else _vl_metric_value <= _vl_best
        )
        if _vl_improved:
            _vl_best = _vl_metric_value
            if _vl_save_best_checkpoint:
                torch.save(net.state_dict(), 'best_model.pt')
        net.train()

.pt dataset: 980 samples, batch_size=64
.pt validation dataset: 280 samples, batch_size=64
{"__nmtk_progress__": true, "epoch": 1, "total_epochs": 700, "loss": 7.764524877071381, "accuracy": null, "layer_spike_rates": {"output": 0.0006417410913854837}}
{"__nmtk_progress__": true, "epoch": 2, "total_epochs": 700, "loss": 6.080155074596405, "accuracy": null, "layer_spike_rates": {"output": 0.0032087054569274187}}
{"__nmtk_progress__": true, "epoch": 3, "total_epochs": 700, "loss": 5.228610515594482, "accuracy": null, "layer_spike_rates": {"output": 0.0020647321362048388}}
{"__nmtk_progress__": true, "epoch": 4, "total_epochs": 700, "loss": 4.987925708293915, "accuracy": null, "layer_spike_rates": {"output": 0.0010323660681024194}}
{"__nmtk_progress__": true, "epoch": 5, "total_epochs": 700, "loss": 5.654028385877609, "accuracy": null, "layer_spike_rates": {"output": 0.0005022321711294353}}
{"__nmtk_progress__": true, "epoch": 6, "total_epochs": 700, "loss": 5.8840674459934235, "accuracy"

## Evaluate

In [8]:
import torch
from torch.serialization import safe_globals
from torch.utils.data import DataLoader, TensorDataset

try:
    with safe_globals([TensorDataset]):
        _raw = torch.load('dataLoader_1784115155942_ds_test.pt', weights_only=True)
except FileNotFoundError as exc:
    raise RuntimeError(
        'Dataset file not found: ' + 'dataLoader_1784115155942_ds_test.pt' + '. '
        'This file does not exist yet — if it is a hand-built stimulus/dataset, generate it first (see the notebook guide\'s data-preparation step), then retry.'
    ) from exc
except Exception as exc:
    raise RuntimeError(
        'Could not safely load trusted .pt dataset ' + 'dataLoader_1784115155942_ds_test.pt' + '. '
        'Expected a TensorDataset, tuple/list of tensors, a bare tensor, or a dict with data/labels. '
        'Regenerate the dataset in one of those formats, then retry.'
    ) from exc
# Handle TensorDataset, dict, raw tuple, or a bare tensor (e.g. an
# unlabeled stimulus/spike-train tensor with no dataset wrapper)
_ds = _raw if hasattr(_raw, 'tensors') else TensorDataset(*_raw) if isinstance(_raw, (tuple, list)) else TensorDataset(_raw) if torch.is_tensor(_raw) else TensorDataset(_raw['data'], _raw['labels'])

train_loader = DataLoader(_ds, batch_size=64, shuffle=True, num_workers=0)
test_loader  = DataLoader(_ds, batch_size=64, shuffle=False,          num_workers=0)
print(f'.pt dataset: {len(_ds)} samples, batch_size=64')
try:
    net.load_state_dict(torch.load('best_model.pt', weights_only=True))
    print('Loaded best checkpoint from best_model.pt')
except FileNotFoundError:
    print("best_model.pt not found — evaluating with current in-memory weights "
          "(enable 'Save Best Checkpoint' on the Validation Loop node and train first).")
correct = 0; total = 0
net.eval()
with torch.no_grad():
    for batch_idx, (data, targets) in enumerate(test_loader):
        # Reset network membrane potentials
        for layer in net.modules():
            if hasattr(layer, 'reset_mem'):
                layer.reset_mem()
        
        spk_out, mem_out = net(data)
        
        correct += (spk_out.sum(0).argmax(1) == targets).sum().item()
        total   += targets.size(0)
        _nmtk_emit(
            epoch=batch_idx + 1,
            total=len(test_loader),
            loss=0.0,
            accuracy=(correct / total if total else None),
            layer_rates={'output': float(spk_out.float().mean().item())},
        )
print(f'Accuracy (top-1): {correct/total:.2%}' if total else 'Accuracy (top-1): n/a')
_nmtk_emit(
    epoch=1,
    total=1,
    loss=0.0,
    accuracy=(correct / total if total else None),
    layer_rates={'output': float(spk_out.float().mean().item())},
)

.pt dataset: 140 samples, batch_size=64
Loaded best checkpoint from best_model.pt
{"__nmtk_progress__": true, "epoch": 1, "total_epochs": 3, "loss": 0.0, "accuracy": 0.8125, "layer_spike_rates": {"output": 0.0702253058552742}}
{"__nmtk_progress__": true, "epoch": 2, "total_epochs": 3, "loss": 0.0, "accuracy": 0.859375, "layer_spike_rates": {"output": 0.0772530660033226}}
{"__nmtk_progress__": true, "epoch": 3, "total_epochs": 3, "loss": 0.0, "accuracy": 0.8642857142857143, "layer_spike_rates": {"output": 0.0823567733168602}}
Accuracy (top-1): 86.43%
{"__nmtk_progress__": true, "epoch": 1, "total_epochs": 1, "loss": 0.0, "accuracy": 0.8642857142857143, "layer_spike_rates": {"output": 0.0823567733168602}}


In [9]:
# ── Download as Python script ──────────────────────────────────
# Run this cell to download the notebook as a .py script.
import subprocess
subprocess.run(['jupyter', 'nbconvert', '--to', 'script',
                '__file__'], check=False)
print('Conversion triggered — check the file listing.')

This application is used to convert notebook files (*.ipynb) to various other
formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the notebook prior to export.
    Equivalent to: [--ExecutePreprocess

[NbConvertApp] WARNING | pattern '__file__' matched no files
